# Musanga V4 — NB02: V3_1-proven RGB U-Net with cached chips, 78-plot spatial CV

**Implementation follows the working V3_1/02B recipe**: pretrained ResNet18 U-Net from `segmentation_models_pytorch`, one-time 256 px chip preparation from a persistent raster handle per mosaic, compressed resumable `.npz` caches, GPU mixed precision, and an in-memory training dataset. Crucially, **there are no repeated Google Drive raster reads inside the training epochs**.

V4 differences are **only** the 78 exhaustive field plots, their crown GPKG, six RGB mosaics, joint plot–circle folds, and the final deduplicated window manifest. Never mix legacy V3_1 checkpoints with this reference.

Start with `SMOKE_TEST=True` (small cache, one fold). Run full 3-fold spatial validation separately.


In [ ]:
# 00 — Environment, frozen inputs and run settings
%pip -q install geopandas pyogrio shapely rasterio scipy scikit-learn torch matplotlib segmentation-models-pytorch
from pathlib import Path
import json, math, random, copy, time, warnings, os
import numpy as np, pandas as pd, geopandas as gpd, rasterio
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from rasterio.windows import Window
from rasterio.features import rasterize
from shapely.geometry import box
from shapely.ops import unary_union
from shapely import make_valid
from shapely.strtree import STRtree
from IPython.display import display
from google.colab import drive
if not Path('/content/drive/MyDrive').exists():drive.mount('/content/drive')
DRIVE=Path('/content/drive/MyDrive')
ROOT=DRIVE/'Colab Notebooks/Musanga/Final'
assert ROOT.is_dir(), ROOT
DATA=ROOT/'Data'
REF_DIR=ROOT/'Outputs/v4/01_reference_field78_combined'
CONTRACT=json.loads((REF_DIR/'01_field78_reference_contract.json').read_text())
assert CONTRACT['schema']=='musanga_v4_field78_combined_reference_final_2',CONTRACT['schema']
NB00=json.loads(Path(CONTRACT['input_manifest']).read_text())
assert NB00['n_field_plots']==78
SPLITS=pd.read_csv(CONTRACT['cv_manifest'])
assert len(SPLITS)==9096,(len(SPLITS),'Expected 3032 unique windows × 3 folds; changed source audit?')
K=['cv_fold','mosaic','row','col','width','height']
assert not SPLITS.duplicated(K).any()
assert SPLITS.groupby('cv_fold').size().eq(3032).all()
assert set(SPLITS.split).issubset({'train','val','excluded_spatial_guard','excluded_no_validation_pixels'})
PLOTS=gpd.read_file(NB00['field_support_qa']).to_crs('EPSG:32635')
PLOT_FOLDS=pd.read_csv(CONTRACT['folds_csv'])
assert len(PLOTS)==len(PLOT_FOLDS)==78 and PLOT_FOLDS.source_id.is_unique
PLOTS=PLOTS.merge(PLOT_FOLDS[['source_id','site_key','cv_fold']],left_on='Plot_name',right_on='source_id',validate='one_to_one')
CIRCLE_FOLDS=pd.read_csv(REF_DIR/'01_field78_circle_folds.csv')
# Adjustable, deterministic experiment settings
SMOKE_TEST=True            # FIRST RUN; switch False for real training
RUN_FOLDS=[0]              # full scientific run: [0,1,2]
SEED=20261113
CHIP=256                   # 256 native pixels; no spatial downsampling
BATCH_SIZE=8
EPOCHS=1 if SMOKE_TEST else 25
STEPS_PER_EPOCH=3 if SMOKE_TEST else 100
EVAL_BATCH_SIZE=8  # batched exhaustive held-out inference, mosaic handle opened once
LEARNING_RATE=2e-4
PATIENCE=4
POSITIVE_SAMPLE_BOOST=2.5
MAX_VAL_WINDOWS_PER_SITE=2 if SMOKE_TEST else 35  # early-stopping cache only; FINAL evaluation is uncapped
MAX_TRAIN_WINDOWS_PER_SITE=8 if SMOKE_TEST else 180
NUM_WORKERS=0              # Google Drive + GDAL: safest in Colab
OUT=ROOT/'Outputs/v4/02_rgb_unet_field78_v3recipe'/('smoke' if SMOKE_TEST else 'full_spatial_cv')
OUT.mkdir(parents=True,exist_ok=True)
random.seed(SEED);np.random.seed(SEED);torch.manual_seed(SEED)
if torch.cuda.is_available():torch.cuda.manual_seed_all(SEED)
DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
assert DEVICE.type=='cuda', 'V3_1 used GPU; select Colab GPU runtime before training'
print('Device:',DEVICE,'| train mode:', 'SMOKE TEST' if SMOKE_TEST else 'FULL SPATIAL CV')
print('Final manifest:',CONTRACT['cv_manifest'])
print('Observed window splits:');display(SPLITS.groupby(['cv_fold','site','split']).size().unstack(fill_value=0))


## 01 — Load and repair vector supervision once; verify exhaustive supports

The 78 plots and 34 circles are separate exhaustive areas. Any circle area inside a field plot is overridden by its field-plot annotation. Outside exhaustive areas, partial-site polygons label **their own pixels only**. Geometries are repaired conservatively (≤5% area shift), preserving source files. An explicit negative pixel is not inferred outside plots/circles.


In [ ]:
# 01 — Geometry references and fast spatial indexes
CROWNS=gpd.read_file(NB00['exhaustive_plot_crowns']).to_crs('EPSG:32635')
CIRCLE_ROI=gpd.read_file(NB00['circle_complete_roi']).to_crs('EPSG:32635')
CIRCLE_POS=gpd.read_file(NB00['circle_positive_crowns']).to_crs('EPSG:32635')
assert len(CROWNS)==3324 and len(CIRCLE_ROI)==34
assert 'Class_ID' in CROWNS
def polygons(g):
    if g is None or g.is_empty:return []
    if g.geom_type=='Polygon':return [g]
    if g.geom_type=='MultiPolygon':return list(g.geoms)
    return [a for v in getattr(g,'geoms',[]) for a in polygons(v)]
def fix(g):
    if g is None or g.is_empty:return None
    if g.is_valid:return g
    q=unary_union(polygons(make_valid(g)))
    if q.is_empty or not q.is_valid:return None
    if abs(q.area-g.area)/max(q.area,g.area,1e-12)>.05:return None
    return q
CROWNS.geometry=[fix(g) for g in CROWNS.geometry]
assert CROWNS.geometry.notna().all(),'Unrepaired plot crowns; see NB00 QA'
CIRCLE_ROI.geometry=[fix(g) for g in CIRCLE_ROI.geometry]
CIRCLE_POS.geometry=[fix(g) for g in CIRCLE_POS.geometry]
assert CIRCLE_ROI.geometry.notna().all()
CIRCLE_ROI['circle_id']=np.arange(len(CIRCLE_ROI))
CIRCLE_ROI=CIRCLE_ROI.merge(CIRCLE_FOLDS[['circle_id','site_key','cv_fold',
    'cross_fold_plot_intersection']],on='circle_id',validate='one_to_one')
CROWN_POS=list(CROWNS.loc[pd.to_numeric(CROWNS.Class_ID).eq(1)].geometry)
CROWN_NEG=list(CROWNS.loc[~pd.to_numeric(CROWNS.Class_ID).eq(1)].geometry)
CIRCLE_POS_GEOMS=[g for g in CIRCLE_POS.geometry if g is not None and not g.is_empty]
# Partial site references from old V3_1 audit, same positive=1 / other known class negative.
PARTIAL={}
for site,path in NB00['partial_site_crowns'].items():
    d=gpd.read_file(path).to_crs('EPSG:32635')
    d.geometry=[fix(g) for g in d.geometry]
    d=d[d.geometry.notna() & pd.to_numeric(d.Class_ID,errors='coerce').notna()]
    PARTIAL[site]=d
P_GEOMS=list(PLOTS.geometry)
C_GEOMS=list(CIRCLE_ROI.geometry)
P_T=STRtree(P_GEOMS); C_T=STRtree(C_GEOMS)
CP_T=STRtree(CROWN_POS);CN_T=STRtree(CROWN_NEG)
KPOS_T=STRtree(CIRCLE_POS_GEOMS)
PARTIAL_T={}
for site,d in PARTIAL.items():
    pos=list(d.loc[pd.to_numeric(d.Class_ID).eq(1)].geometry)
    neg=list(d.loc[~pd.to_numeric(d.Class_ID).eq(1)].geometry)
    PARTIAL_T[site]=(pos,neg,STRtree(pos) if pos else None,
                     STRtree(neg) if neg else None)
RGB={}
for item in sorted(SPLITS.mosaic.unique()):
    q=DATA/'Ortho_RGB'/item
    assert q.exists(),q
    with rasterio.open(q) as ds:
        assert ds.crs.to_epsg()==32635,(item,ds.crs)
        assert ds.count>=3,(item,ds.count)
        RGB[item]=dict(path=str(q),transform=ds.transform,site=next(
            iter(SPLITS.loc[SPLITS.mosaic==item,'site'].unique())))
# NOTE: ROI.shp is the authoritative six-mosaic modelling ROI (not the old
# circle-only support). The validation/labelled field plot is also required.
ROIS=gpd.read_file(DATA/'Ortho_RGB/ROI.shp').to_crs('EPSG:32635')
ROI_GEOMS=list(ROIS.geometry)
ROI_T=STRtree(ROI_GEOMS)
print('Loaded',len(PLOTS),'plots,',len(CIRCLE_ROI),'circles,',
      len(CROWN_POS),'Musanga plot crowns and',len(CIRCLE_POS_GEOMS),'circle crowns')


In [ ]:
# 02 — Rasterize strict plot > circle > partial supervision on demand
# Geometry-level spatial queries restrict operations to the local 256px footprint.
def geom_hits(tree,geoms,ext):
    if tree is None:return []
    return [geoms[int(i)] for i in tree.query(ext,predicate='intersects')]
_ACTIVE_RASTER=None
_ACTIVE_NAME=None
def rgb_window(raster_path,win):
    from contextlib import nullcontext
    name=Path(raster_path).name
    ds_context=nullcontext(_ACTIVE_RASTER) if (_ACTIVE_RASTER is not None and _ACTIVE_NAME==name) else rasterio.open(raster_path)
    with ds_context as ds:
        a=ds.read([1,2,3],window=win,masked=True)
        valid=ds.dataset_mask(window=win)>0
        valid &= ~np.ma.getmaskarray(a).any(axis=0)
        x=np.asarray(a.filled(0),dtype=np.float32)
        # RGB products expected standard 8-bit; prevent silent incorrect scaling.
        if np.issubdtype(np.dtype(ds.dtypes[0]),np.integer) and np.dtype(ds.dtypes[0]).itemsize==1:
            x=x/255.
        else:
            raise RuntimeError(f'Non-uint8 RGB {ds.dtypes}: {raster_path}; define radiometric scaling first')
    return x,np.asarray(valid,bool)
def supervision(mosaic,win,fold,role):
    site=RGB[mosaic]['site'];t=RGB[mosaic]['transform']
    h,w=int(win.height),int(win.width)
    affine=rasterio.windows.transform(win,t)
    footprint=box(*rasterio.windows.bounds(win,t))
    def burn(gs):
        return rasterize(((g,1) for g in gs if g is not None and not g.is_empty),
            out_shape=(h,w),transform=affine,fill=0,dtype='uint8').astype(bool) if gs else np.zeros((h,w),bool)
    roi=burn(geom_hits(ROI_T,ROI_GEOMS,footprint))
    pids=[int(i) for i in P_T.query(footprint,predicate='intersects')
          if PLOTS.iloc[int(i)].site_key==site and
          (role=='train' and PLOTS.iloc[int(i)].cv_fold!=fold or
           role=='val' and PLOTS.iloc[int(i)].cv_fold==fold)]
    plot=burn([P_GEOMS[i] for i in pids])
    # Every other fold's field plot is out of scope for val; all exhaustive
    # plot geometries (not only selected) override circle labels.
    allplot=burn([P_GEOMS[int(i)] for i in P_T.query(footprint,predicate='intersects')
                  if PLOTS.iloc[int(i)].site_key==site])
    cids=[int(i) for i in C_T.query(footprint,predicate='intersects')
          if CIRCLE_ROI.iloc[int(i)].site_key==site and
          not CIRCLE_ROI.iloc[int(i)].cross_fold_plot_intersection and
          (role=='train' and CIRCLE_ROI.iloc[int(i)].cv_fold!=fold)]
    circle=burn([C_GEOMS[i] for i in cids])&~allplot if role=='train' else np.zeros((h,w),bool)
    posplot=burn(geom_hits(CP_T,CROWN_POS,footprint))&plot
    negplot=burn(geom_hits(CN_T,CROWN_NEG,footprint))&plot
    poscircle=burn(geom_hits(KPOS_T,CIRCLE_POS_GEOMS,footprint))&circle
    part_pos=np.zeros((h,w),bool);part_neg=np.zeros((h,w),bool)
    if role=='train':
        positives,negatives,tp,tn=PARTIAL_T[site]
        part_pos=burn(geom_hits(tp,positives,footprint))&~allplot&~circle
        part_neg=burn(geom_hits(tn,negatives,footprint))&~allplot&~circle
    target=(posplot|poscircle|part_pos)
    conflicting=(posplot&negplot)|(part_pos&part_neg)
    weight=roi&(plot|circle|part_pos|part_neg)&~conflicting
    return target.astype(np.float32),weight.astype(np.float32),plot.astype(bool)
def get_chip(mosaic,win,fold,role):
    x,rgbvalid=rgb_window(RGB[mosaic]['path'],win)
    y,w,plot=supervision(mosaic,win,fold,role)
    w*=rgbvalid.astype(np.float32)
    return x,y,w,plot
print('Prepared lazy RGB + spatially strict masks. No imagery loaded globally.')


## 02 — V3_1 chip preparation and resumable cache (once per spatial fold)

As in V3_1, select a balanced subset of **physical windows**, group by mosaic, open each image once, extract **256 × 256 RGB + label + loss-mask chips**, and write compressed uint8 NPZ chunks. On rerun the fingerprint permits a **cache hit**. During GPU training, the dataset reads only RAM arrays—not Google Drive rasters. Small smoke settings cap the number of source windows per site.


In [ ]:
# 03 — V3_1-style resumable cache, file provenance, persistent raster handles
from contextlib import contextmanager
import hashlib, shutil
from collections import defaultdict
from rasterio.windows import Window
CACHE_DIR=OUT/'chip_cache';CACHE_DIR.mkdir(parents=True,exist_ok=True)
USE_LOCAL_STAGE=False  # Same optional one-mosaic SSD staging as V3_1; disabled by default
STAGE_DIR=Path('/content/musanga_v4_stage');STAGE_DIR.mkdir(parents=True,exist_ok=True)
@contextmanager
def staged_mosaic(name):
    global _ACTIVE_RASTER,_ACTIVE_NAME
    src=Path(RGB[name]['path']);selected=src
    local=STAGE_DIR/src.name
    if USE_LOCAL_STAGE and shutil.disk_usage('/content').free>src.stat().st_size+2*1024**3:
        if not local.exists() or local.stat().st_size!=src.stat().st_size:
            tmp=local.with_name(local.name+'.part')
            with src.open('rb') as a,tmp.open('wb') as b:
                shutil.copyfileobj(a,b,16*1024*1024)
            tmp.replace(local)
        selected=local
    assert _ACTIVE_RASTER is None
    try:
        with rasterio.open(selected) as ds:
            assert ds.transform==RGB[name]['transform'] and ds.count>=3
            _ACTIVE_RASTER=ds;_ACTIVE_NAME=name
            yield
    finally:
        _ACTIVE_RASTER=None;_ACTIVE_NAME=None
        if selected==local:local.unlink(missing_ok=True)
def cache_signature(fold,role):
    dependencies=[Path(CONTRACT['cv_manifest']),Path(CONTRACT['input_manifest']),
        Path(NB00['exhaustive_plot_crowns']),Path(NB00['plot_file']),
        Path(NB00['circle_positive_crowns']),Path(NB00['circle_complete_roi'])]
    dependencies += [Path(p) for p in NB00['partial_site_crowns'].values()]
    source_state=[(str(p),p.stat().st_size,p.stat().st_mtime_ns) for p in dependencies]
    info=dict(version='v4_resnet18_v3cache_v1',fold=int(fold),role=role,
        seed=SEED,chip=CHIP,smoke=SMOKE_TEST,src=source_state,
        max_train=MAX_TRAIN_WINDOWS_PER_SITE,max_val=MAX_VAL_WINDOWS_PER_SITE)
    return hashlib.sha256(json.dumps(info,sort_keys=True).encode()).hexdigest()
def pack(path,items,signature):
    tmp=path.with_name(path.name+'.tmp.npz')
    np.savez_compressed(tmp,key=np.array(signature),
        rgb=np.stack([r['rgb'] for r in items]).astype(np.uint8),
        y=np.stack([r['y'] for r in items]).astype(np.uint8),
        w=np.stack([r['w'] for r in items]).astype(np.uint8),
        site=np.array([r['site'] for r in items],dtype='U20'),
        mosaic=np.array([r['mosaic'] for r in items],dtype='U100'),
        row=np.array([r['row'] for r in items],dtype=np.int32),
        col=np.array([r['col'] for r in items],dtype=np.int32))
    tmp.replace(path)
def unpack(path,signature):
    with np.load(path,allow_pickle=False) as z:
        assert str(z['key'].item())==signature
        return [dict(rgb=z['rgb'][i],y=z['y'][i],w=z['w'][i],
            site=str(z['site'][i]),mosaic=str(z['mosaic'][i]),
            row=int(z['row'][i]),col=int(z['col'][i])) for i in range(len(z['rgb']))]
def source_window_rows(fold,role):
    q=SPLITS[(SPLITS.cv_fold==fold)&(SPLITS.split==role)].copy()
    assert len(q)>0,(fold,role)
    cap=MAX_TRAIN_WINDOWS_PER_SITE if role=='train' else MAX_VAL_WINDOWS_PER_SITE
    q['likely_positive']=q.apply(lambda r:bool(
        CP_T.query(box(*rasterio.windows.bounds(
            Window(int(r.col),int(r.row),int(r.width),int(r.height)),
            RGB[r.mosaic]['transform'])),predicate='intersects').size),axis=1)
    selected=[]
    for site,g in q.groupby('site',sort=True):
        # Approximate stratification by positive intersection before any image reads.
        high=g[g.likely_positive];low=g[~g.likely_positive]
        if role=='train' and len(high) and len(low):
            nh=min(len(high),int(cap*.65))
            chosen=pd.concat([high.sample(n=nh,random_state=SEED+fold),
                 low.sample(n=min(len(low),cap-nh),random_state=SEED+fold)])
        else:
            chosen=g.sample(n=min(cap,len(g)),random_state=SEED+fold)
        selected.append(chosen)
    return pd.concat(selected,ignore_index=True)
def make_cache(fold,role):
    sig=cache_signature(fold,role)
    cache=CACHE_DIR/f'field78_{role}_fold{fold}_{sig[:16]}.npz'
    if cache.exists():
        try:
            items=unpack(cache,sig)
            print('CACHE HIT',role,'fold',fold,'n',len(items),flush=True)
            return items
        except Exception as exc:
            print('Invalid cache:',exc)
            cache.unlink(missing_ok=True)
    selected=source_window_rows(fold,role)
    print('PREPARE',role,'fold',fold,
        selected.groupby('site').size().to_dict(),flush=True)
    parts_dir=CACHE_DIR/f'{role}_fold{fold}_{sig[:16]}'
    parts_dir.mkdir(exist_ok=True)
    items=[]
    t0=time.perf_counter()
    for mosaic,g in selected.groupby('mosaic',sort=True):
        with staged_mosaic(mosaic):
            for k in range(0,len(g),50):
                part=parts_dir/f'{mosaic}_{k:04d}.npz'
                if part.exists():
                    try:
                        existing=unpack(part,sig)
                        items.extend(existing);continue
                    except Exception:part.unlink(missing_ok=True)
                batch=[]
                for r in list(g.iloc[k:k+50].itertuples(index=False)):
                    # Same deterministic centre crop as V3_1; do not read 512px
                    # merely to throw away 3/4 of the source pixel data.
                    if r.width<CHIP or r.height<CHIP:continue
                    row=int(r.row)+(int(r.height)-CHIP)//2
                    col=int(r.col)+(int(r.width)-CHIP)//2
                    win=Window(col,row,CHIP,CHIP)
                    x,y,w,_=get_chip(mosaic,win,fold,role)
                    if w.sum()<50:continue
                    batch.append(dict(rgb=np.rint(np.clip(x.transpose(1,2,0),0,1)*255).astype('uint8'),
                        y=y.astype('uint8'),w=w.astype('uint8'),
                        site=RGB[mosaic]['site'],mosaic=mosaic,row=row,col=col))
                if batch:
                    pack(part,batch,sig)
                    items.extend(batch)
                print('Cached',role,'fold',fold,mosaic,
                    min(k+50,len(g)),'/',len(g),'usable',len(items),
                    'elapsed_s',round(time.perf_counter()-t0,1),flush=True)
    assert len(items)>0,(fold,role,'No valid RGB+reference chips')
    pack(cache,items,sig)
    print('CACHE SAVED',role,'fold',fold,'chips',len(items),
        'seconds',round(time.perf_counter()-t0,1),flush=True)
    return items


In [ ]:
# 04 — V3_1-style RAM-backed dataset (no GDAL calls inside GPU training)
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
class CachedChips(Dataset):
    def __init__(self,items,augment=False):
        self.items=items;self.augment=augment
    def __len__(self):return len(self.items)
    def __getitem__(self,i):
        q=self.items[i]
        rgb=q['rgb'];y=q['y'];w=q['w']
        if self.augment:
            k=int(torch.randint(0,4,(1,)).item())
            rgb=np.rot90(rgb,k);y=np.rot90(y,k);w=np.rot90(w,k)
            if torch.rand(1).item()<.5:
                rgb=rgb[:,::-1];y=y[:,::-1];w=w[:,::-1]
        x=torch.from_numpy(np.ascontiguousarray(rgb.transpose(2,0,1))).float()/255.
        return x,torch.from_numpy(np.ascontiguousarray(y[None])).float(),torch.from_numpy(np.ascontiguousarray(w[None])).float(),q['site']
def cached_sampler(items):
    counts=pd.Series([r['site'] for r in items]).value_counts().to_dict()
    weights=[(POSITIVE_SAMPLE_BOOST if np.any(r['y']&r['w']) else 1.)/counts[r['site']]
             for r in items]
    return WeightedRandomSampler(torch.as_tensor(weights,dtype=torch.double),
        num_samples=min(len(items),STEPS_PER_EPOCH*BATCH_SIZE),replacement=True)
print('Training uses 100% RAM-backed V3_1-style cached chips.')


## 03 — V3_1 training with cached chips and efficient model selection

Train the original pretrained ResNet18 U-Net on the sampled/resumable chip cache. The checkpoint-selection validation cache is a **site-stratified sample**, deliberately limited for speed, and is **not** the scientific evaluation. The final evaluation in Section 04 reads *every* held-out field-plot window and every non-overlapping 256-pixel tile (including edge tiles), with no cap, and reports completeness per plot. GPU inference is batched and image handles remain open per mosaic, as in V3_1.


In [ ]:
# 05 — Restore the working V3_1 ResNet18 U-Net + masked BCE, AMP in training
import segmentation_models_pytorch as smp
import torch.nn.functional as F
class ResNet18UNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.net=smp.Unet(encoder_name='resnet18',encoder_weights='imagenet',
            in_channels=3,classes=1)
    def forward(self,x):return self.net(x)  # B,1,H,W
def masked_loss(logits,y,weight):
    bce=F.binary_cross_entropy_with_logits(logits.float(),y.float(),reduction='none')
    return (bce*weight).sum()/weight.sum().clamp_min(1.)
AMP=DEVICE.type=='cuda'
def run_epoch(model,loader,optimizer=None,scaler=None):
    model.train(optimizer is not None)
    totals=[];site_losses={}
    for x,y,w,site in loader:
        x=x.to(DEVICE,non_blocking=True);y=y.to(DEVICE,non_blocking=True)
        w=w.to(DEVICE,non_blocking=True)
        if not w.any():continue
        with torch.set_grad_enabled(optimizer is not None):
            with torch.autocast(device_type='cuda',dtype=torch.float16,enabled=AMP):
                logits=model(x)
            loss=masked_loss(logits,y,w)
            if optimizer is not None:
                optimizer.zero_grad(set_to_none=True)
                scaler.scale(loss).backward()
                scaler.step(optimizer);scaler.update()
        totals.append(loss.detach().item())
        if optimizer is None:
            # Per-chip, equal-site loss (avoid ForestGEO pixel-volume dominance)
            for i,name in enumerate(site):
                if w[i].sum()==0:continue
                l=masked_loss(logits[i:i+1],y[i:i+1],w[i:i+1])
                site_losses.setdefault(str(name),[]).append(float(l.item()))
    if optimizer is None:
        return float(np.mean([np.mean(v) for v in site_losses.values()])) if site_losses else float('inf')
    return float(np.mean(totals)) if totals else float('inf')


In [ ]:
# 06 — Prepare each fold once, then reuse precomputed chips for every epoch
HISTORY=[];CHECKPOINTS={}
for fold in RUN_FOLDS:
    assert fold in (0,1,2)
    t0=time.perf_counter()
    train_chips=make_cache(fold,'train')
    val_chips=make_cache(fold,'val')
    print('Total cache loading/preparation seconds:',round(time.perf_counter()-t0,1))
    trainloader=DataLoader(CachedChips(train_chips,augment=True),
        batch_size=BATCH_SIZE,sampler=cached_sampler(train_chips),
        num_workers=0,pin_memory=AMP)
    valloader=DataLoader(CachedChips(val_chips,augment=False),
        batch_size=BATCH_SIZE,shuffle=False,num_workers=0,pin_memory=AMP)
    model=ResNet18UNet().to(DEVICE)
    optimizer=torch.optim.AdamW(model.parameters(),lr=LEARNING_RATE,weight_decay=1e-4)
    scaler=torch.amp.GradScaler('cuda',enabled=AMP)
    best=float('inf');wait=0
    ckpt=OUT/f'V4_RGB_ResNet18_fold{fold}_best.pt'
    for epoch in range(1,EPOCHS+1):
        tic=time.perf_counter()
        trloss=run_epoch(model,trainloader,optimizer,scaler)
        valloss=run_epoch(model,valloader,None,None)
        item=dict(fold=fold,epoch=epoch,train_loss=trloss,
            val_loss_site_macro=valloss,epoch_seconds=round(time.perf_counter()-tic,1),
            n_train_chips=len(train_chips),n_val_chips=len(val_chips))
        HISTORY.append(item);print(item,flush=True)
        if valloss<best-1e-4:
            best=valloss;wait=0
            torch.save(dict(model=model.state_dict(),fold=fold,
                architecture='smp_unet_resnet18_imagenet',smoke=SMOKE_TEST,
                epoch=epoch,val_loss=best,cache_recipe='V3_1_resumable_npz'),ckpt)
        else:
            wait+=1
            if wait>=PATIENCE:break
    CHECKPOINTS[fold]=ckpt
    del train_chips,val_chips,trainloader,valloader,model
    if AMP:torch.cuda.empty_cache()
pd.DataFrame(HISTORY).to_csv(OUT/'02_rgb_training_history.csv',index=False)
print('Best V4 checkpoints:',CHECKPOINTS)


## 04 — Exhaustive held-out evaluation, all plots and all valid pixels

For `SMOKE_TEST=False`, process **every window designated `val`** in each fold of the frozen final NB01 manifest. Split windows into non-overlapping 256 × 256 tiles; pad edge tiles to 256 for inference and crop predictions back to their original sizes. Reuse one open raster handle per mosaic and batch inference on GPU. Count each rasterized held-out field plot separately, using only valid RGB and ROI pixels; preserve zero-positive plots for false-positive rate and cover error. Report per-plot and equal-site statistics and **check whether each plot has actually been evaluated**. During smoke mode only, evaluation intentionally remains limited to two windows per site.

This is **not** an independent external validation (all sites contribute other plots to training). It is spatially blocked within-site CV.


### 06B — Restore the three FULL checkpoints without training
**If restarting Colab:** run setup Sections 00–05, **skip Section 06 training**, then run this Section 06B before Sections 07–09. It explicitly switches out of the smoke output directory and verifies all three checkpoints. Do not use smoke outputs for scientific conclusions.


In [ ]:
# 06B — EVALUATION-ONLY restore: run after setup Sections 00–05; SKIP training Section 06.
# This cell intentionally overrides a leftover SMOKE_TEST=True session.
SMOKE_TEST=False
RUN_FOLDS=[0,1,2]
OUT=ROOT/'Outputs/v4/02_rgb_unet_field78_v3recipe/full_spatial_cv'
assert OUT.is_dir(),f'Full training output directory missing: {OUT}'
CHECKPOINTS={fold:OUT/f'V4_RGB_ResNet18_fold{fold}_best.pt'
             for fold in RUN_FOLDS}
for fold,path in CHECKPOINTS.items():
    assert path.is_file(),f'Missing FULL checkpoint for fold {fold}: {path}'
    metadata=torch.load(path,map_location='cpu',weights_only=False)
    assert int(metadata.get('fold',-1))==fold,(fold,metadata.get('fold'))
    assert metadata.get('smoke') is False,(
        f'Fold {fold} is a SMOKE checkpoint; refusing scientific evaluation')
    assert metadata.get('architecture')=='smp_unet_resnet18_imagenet'
    print(f'FULL checkpoint fold {fold}: epoch {metadata.get("epoch")}, '
          f'checkpoint validation loss {metadata.get("val_loss"):.4f}')
    del metadata
print('EVALUATION-ONLY: three full CV checkpoints verified. Run Sections 07, 08, 09.')


In [ ]:
# 07 — Efficient *exhaustive* held-out evaluation (V3_1 persistent raster + GPU batches)
assert not SMOKE_TEST and len(RUN_FOLDS)==3 and set(RUN_FOLDS)=={0,1,2}, (
    'Scientific report requires FULL three-fold checkpoints. Run Section 06B.')
for _fold in RUN_FOLDS:
    _c=CHECKPOINTS.get(_fold)
    assert _c is not None and Path(_c).exists(), ('Missing full checkpoint',_fold)
    _meta=torch.load(_c,map_location='cpu',weights_only=False)
    assert _meta.get('smoke') is False and _meta.get('fold')==_fold, (
        'Refusing smoke/wrong-fold checkpoint',_fold)
from rasterio.windows import Window
from shapely.geometry import box
from rasterio.features import rasterize

def burn_single(g,win,affine):
    return rasterize([(g,1)],out_shape=(int(win.height),int(win.width)),
                     transform=affine,fill=0,dtype='uint8').astype(bool)

def full_validation_windows(fold):
    q=SPLITS.loc[(SPLITS.cv_fold==fold)&SPLITS.split.eq('val')].copy()
    assert len(q) and not q.duplicated(['mosaic','row','col','width','height']).any()
    if SMOKE_TEST:
        return q.groupby('site',group_keys=False).head(MAX_VAL_WINDOWS_PER_SITE).copy()
    # Important: no sampling, capping or preferential exclusion in full evaluation.
    expected=int(((SPLITS.cv_fold==fold)&(SPLITS.split=='val')).sum())
    assert len(q)==expected
    return q

def eval_tiles(q):
    for r in q.itertuples(index=False):
        for dy in range(0,int(r.height),CHIP):
            for dx in range(0,int(r.width),CHIP):
                hh=min(CHIP,int(r.height)-dy);ww=min(CHIP,int(r.width)-dx)
                if hh>0 and ww>0:
                    yield r.mosaic,Window(int(r.col)+dx,int(r.row)+dy,ww,hh),str(r.site)

# V3_1-compatible probability histograms (approximate threshold-free metrics).
# 512 bins prevent having to save millions of individual pixel probabilities.
N_BINS=512
BIN_EDGES=np.linspace(0.,1.,N_BINS+1)
HISTS={}
PLOT_ROWS=[]
EVAL_TIMINGS=[]
for fold in RUN_FOLDS:
    ck=torch.load(CHECKPOINTS[fold],map_location=DEVICE,weights_only=False)
    assert ck.get('fold')==fold
    model=ResNet18UNet().to(DEVICE)
    model.load_state_dict(ck['model'])
    model.eval()
    fval=full_validation_windows(fold)
    plot_state={}
    for r in PLOTS.loc[PLOTS.cv_fold==fold].itertuples(index=False):
        plot_state[str(r.Plot_name)]=dict(tp=0,fp=0,fn=0,tn=0,n=0,
                                         site=r.site_key,ba=float(r.pct_BA_musanga),
                                         sum_prob=0.,sum_brier=0.)
    fold_start=time.perf_counter()
    tile_counts=0
    valid_tile_counts=0
    # Group by mosaic, keep raster handle open throughout all its tiles.
    for mosaic,g in fval.groupby('mosaic',sort=True):
        current=[]
        with staged_mosaic(mosaic), torch.no_grad():
            def flush_batch(pending):
                # Each batch contains RGB arrays plus matching masks/plot IDs.
                # Save no raster arrays on GPU after inference.
                if not pending:return
                batch=torch.from_numpy(np.stack([x['rgb'] for x in pending])).to(
                    DEVICE,non_blocking=True)
                with torch.autocast(device_type='cuda',dtype=torch.float16,enabled=AMP):
                    probs=torch.sigmoid(model(batch))[:,0].float().cpu().numpy()
                for item,pred_padded in zip(pending,probs):
                    h,w=item['shape']
                    pred=pred_padded[:h,:w]>=.5
                    obs=item['target']
                    for name,mask in item['plot_masks']:
                        entry=plot_state[name]
                        tp=int(np.count_nonzero(mask&pred&obs))
                        fp=int(np.count_nonzero(mask&pred&~obs))
                        fn=int(np.count_nonzero(mask&~pred&obs))
                        tn=int(np.count_nonzero(mask&~pred&~obs))
                        entry['tp']+=tp;entry['fp']+=fp
                        entry['fn']+=fn;entry['tn']+=tn
                        entry['n']+=int(mask.sum())
                        pv=pred_padded[:h,:w][mask].astype(np.float64)
                        yy=obs[mask].astype(np.uint8)
                        entry['sum_prob']+=float(pv.sum())
                        entry['sum_brier']+=float(np.square(pv-yy).sum())
                        key=(int(fold),name)
                        if key not in HISTS:
                            HISTS[key]=(np.zeros(N_BINS,dtype=np.int64),
                                        np.zeros(N_BINS,dtype=np.int64))
                        ix=np.minimum((pv*N_BINS).astype(np.int32),N_BINS-1)
                        HISTS[key][0][:]+=np.bincount(ix,minlength=N_BINS).astype(np.int64)
                        HISTS[key][1][:]+=np.bincount(ix,weights=yy,
                                                 minlength=N_BINS).astype(np.int64)
            for name,win,site in eval_tiles(g):
                tile_counts+=1
                # This reads only the local window from the already-open mosaic.
                x,valid=rgb_window(RGB[name]['path'],win)
                y,wgt,_=supervision(name,win,fold,'val')
                usable=valid&wgt.astype(bool)
                if not usable.any():continue
                transform=rasterio.windows.transform(win,RGB[name]['transform'])
                footprint=box(*rasterio.windows.bounds(win,RGB[name]['transform']))
                plot_masks=[]
                for i in P_T.query(footprint,predicate='intersects'):
                    p=PLOTS.iloc[int(i)]
                    if int(p.cv_fold)!=fold or p.site_key!=site:continue
                    mask=usable&burn_single(p.geometry,win,transform)
                    if mask.any():
                        plot_masks.append((str(p.Plot_name),mask))
                if not plot_masks:continue
                hh,ww=int(win.height),int(win.width)
                if hh<CHIP or ww<CHIP:
                    x=np.pad(x,((0,0),(0,CHIP-hh),(0,CHIP-ww)),
                             mode='constant',constant_values=0)
                current.append(dict(rgb=x,target=y.astype(bool),
                    plot_masks=plot_masks,shape=(hh,ww)))
                valid_tile_counts+=1
                if len(current)>=EVAL_BATCH_SIZE:
                    flush_batch(current)
                    current=[]
            flush_batch(current)
    row=dict(fold=int(fold),val_windows=int(len(fval)),tiles_examined=tile_counts,
             tiles_with_valid_reference=valid_tile_counts,
             elapsed_seconds=round(time.perf_counter()-fold_start,1))
    EVAL_TIMINGS.append(row)
    print('HELD-OUT EVALUATION:',row,flush=True)
    for name,d in plot_state.items():
        tp,fp,fn,tn=d['tp'],d['fp'],d['fn'],d['tn']
        num=d['n'];pos=tp+fn;predpos=tp+fp
        PLOT_ROWS.append(dict(fold=fold,Plot_name=name,site=d['site'],
            n_rgb_valid_label_pixels=num,field_BA_pct=d['ba'],
            tp=tp,fp=fp,fn=fn,tn=tn,
            crown_reference_pct=100*pos/num if num else np.nan,
            predicted_cover_pct=100*predpos/num if num else np.nan,
            predicted_probability_cover_pct=100*d['sum_prob']/num if num else np.nan,
            brier_score=d['sum_brier']/num if num else np.nan,
            cover_bias_pp=100*(predpos-pos)/num if num else np.nan,
            cover_abs_error_pp=100*abs(predpos-pos)/num if num else np.nan,
            pixel_precision=tp/(tp+fp) if tp+fp else np.nan,
            pixel_recall=tp/pos if pos else np.nan,
            pixel_F1=2*tp/(2*tp+fp+fn) if pos and 2*tp+fp+fn else np.nan,
            pixel_IoU=tp/(tp+fp+fn) if pos and tp+fp+fn else np.nan,
            false_positive_fraction_pct=100*fp/num if num else np.nan))
    del model
    if AMP:torch.cuda.empty_cache()

RESULTS=pd.DataFrame(PLOT_ROWS)
RESULTS.to_csv(OUT/'02_rgb_heldout_per_plot.csv',index=False)
# Cache probability histograms: subsequent metric/figure revisions need no inference.
np.savez_compressed(OUT/'02_rgb_heldout_probability_histograms_512bins.npz',
    **{f'fold{fold}_{name}_count':arr[0] for (fold,name),arr in HISTS.items()},
    **{f'fold{fold}_{name}_positive':arr[1] for (fold,name),arr in HISTS.items()})
SUMMARY=RESULTS.groupby(['fold','site'],as_index=False).agg(
    n_plots=('Plot_name','size'),
    plots_with_evaluation=('n_rgb_valid_label_pixels',lambda x:int((x>0).sum())),
    mean_cover_MAE_pp=('cover_abs_error_pp','mean'),
    mean_cover_bias_pp=('cover_bias_pp','mean'),
    mean_pixel_IoU=('pixel_IoU','mean'),
    mean_pixel_F1=('pixel_F1','mean'),
    mean_false_positive_fraction_pct=('false_positive_fraction_pct','mean'))
SUMMARY.to_csv(OUT/'02_rgb_heldout_by_fold_site.csv',index=False)
pd.DataFrame(EVAL_TIMINGS).to_csv(OUT/'02_rgb_evaluation_timings.csv',index=False)
display(SUMMARY.round(3))
# Diagnose *coverage* versus model performance, before declaring CV complete.
# Keep every plot row even if it has no eligible image pixels.
missing=RESULTS.loc[RESULTS.n_rgb_valid_label_pixels.eq(0),
                   ['fold','site','Plot_name']].copy()
if not missing.empty:
    diag=[]
    for r in missing.itertuples(index=False):
        p=PLOTS.loc[PLOTS.Plot_name.eq(r.Plot_name)].iloc[0]
        geom=p.geometry
        image_hits=[];roi_hits=[];manifest_hits=[]
        for mosaic,meta in RGB.items():
            if meta['site']!=r.site:continue
            with rasterio.open(meta['path']) as ds:
                extent=box(*ds.bounds)
                if geom.intersection(extent).area>0:
                    image_hits.append(mosaic)
                else:
                    continue
                rois=geom_hits(ROI_T,ROI_GEOMS,geom)
                roi_area=unary_union(rois).intersection(geom).intersection(extent).area if rois else 0.
                if roi_area>0:roi_hits.append(mosaic)
            sub=SPLITS.loc[
                (SPLITS.cv_fold==r.fold)&(SPLITS.mosaic==mosaic)&(SPLITS.split=='val')]
            for q in sub.itertuples(index=False):
                win=Window(int(q.col),int(q.row),int(q.width),int(q.height))
                if geom.intersection(box(*rasterio.windows.bounds(
                        win,RGB[mosaic]['transform']))).area>0:
                    manifest_hits.append(mosaic);break
        reason=('outside_all_site_rgb_footprints' if not image_hits else
                'outside_roi_on_rgb_footprints' if not roi_hits else
                'no_heldout_val_window_intersects_plot' if not manifest_hits else
                'inside_manifest_but_no_valid_rgb_roi_reference_pixels')
        diag.append(dict(fold=int(r.fold),site=r.site,Plot_name=r.Plot_name,
            mapped_area_ha=float(geom.area/1e4),reason=reason,
            rgb_footprint_mosaics=';'.join(image_hits),
            roi_covered_mosaics=';'.join(roi_hits),
            intersecting_val_mosaics=';'.join(manifest_hits)))
    MISSING_DIAG=pd.DataFrame(diag)
    MISSING_DIAG.to_csv(OUT/'02_rgb_missing_validation_plot_diagnostics.csv',index=False)
    display(MISSING_DIAG)
    print('WARNING: These plots CANNOT count toward evaluated plot metrics.')
# Status is explicit, never silently claim full coverage.
all_plots_covered=bool(RESULTS.n_rgb_valid_label_pixels.gt(0).all())
observed=int(RESULTS.n_rgb_valid_label_pixels.gt(0).sum())
print(f'VALIDATION COVERAGE: {observed}/{len(RESULTS)} plots with valid evaluated RGB/ROI pixels')
if not SMOKE_TEST:
    assert set(RESULTS.fold)=={0,1,2},'Full CV requires RUN_FOLDS=[0,1,2]'
    assert len(RESULTS)==78 and RESULTS.Plot_name.nunique()==78
    if all_plots_covered:
        print('COMPLETE: all 78 field plots evaluated once.')
    else:
        print(f'INCOMPLETE SPATIAL COVERAGE: {observed}/78 plots evaluated; '
              'report metrics for observed plots only, preserve all 78 records with NaN '
              'for the non-evaluable plots. See missing-plot diagnostics.')
else:
    print('SMOKE TEST ONLY: deliberately sparse evaluation; not scientific accuracy.')
print('Equal-site MAE (each site-fold equally weighted):',
    round(float(SUMMARY.mean_cover_MAE_pp.mean()),3))
RUN_INFO=dict(schema='musanga_v4_rgb_unet_field78_exhaustive_eval_2',
    smoke_test=SMOKE_TEST,folds=RUN_FOLDS,epochs=EPOCHS,
    patches_per_epoch=STEPS_PER_EPOCH*BATCH_SIZE,chip=CHIP,
    final_manifest=CONTRACT['cv_manifest'],threshold=.5,
    checkpoint_selection='sampled cache, macro site validation loss',
    final_evaluation='all held-out plot windows, all tiles (including edges), RGB+ROI validity',
    macro_rule='mean per plot then site; zero-positive F1 undefined',
    all_plots_evaluated=all_plots_covered,
    n_observed_evaluable_plots=observed,
    n_field_plots_total=len(RESULTS),
    validation_coverage_status='complete' if all_plots_covered else 'incomplete_rgb_roi_coverage',
    checkpoints={str(f):str(p) for f,p in CHECKPOINTS.items()},
    model_status='trained_smoke_only' if SMOKE_TEST else 'trained_full_spatial_cv_with_coverage_audit')
(OUT/'02_run_contract.json').write_text(json.dumps(RUN_INFO,indent=2))


## 05 — Comprehensive V3_1-compatible performance report
No retraining. Section 07 now stores 512-bin probability histograms and mean predicted probability for each held-out plot; Section 08 computes classification and cover metrics using those saved outputs. **ROC-AUC and average precision are histogram approximations** (512 bins), not exact pixel-wise ranking. **Comparison with V3_1 is descriptive until identical held-out areas and reference masks are evaluated.**


In [ ]:
# 08 — V3_1-compatible plot metrics: R², MAE, RMSE, RPD, RPIQ, ROC-AUC, AP
assert not SMOKE_TEST and len(RUN_FOLDS)==3 and set(RUN_FOLDS)=={0,1,2}, (
    'Scientific report requires FULL three-fold checkpoints. Run Section 06B.')
for _fold in RUN_FOLDS:
    _c=CHECKPOINTS.get(_fold)
    assert _c is not None and Path(_c).exists(), ('Missing full checkpoint',_fold)
    _meta=torch.load(_c,map_location='cpu',weights_only=False)
    assert _meta.get('smoke') is False and _meta.get('fold')==_fold, (
        'Refusing smoke/wrong-fold checkpoint',_fold)
# Run after Section 07. Reads stored histograms; never retrains the model.
import matplotlib.pyplot as plt
from sklearn.metrics import r2_score, roc_auc_score, average_precision_score
from scipy.stats import pearsonr
REPORT=OUT/'performance_report';REPORT.mkdir(parents=True,exist_ok=True)
R=RESULTS[RESULTS.n_rgb_valid_label_pixels.gt(0)].copy()
assert len(R)==76 if not SMOKE_TEST else len(R)>0
assert R.Plot_name.is_unique
HIST_PATH=OUT/'02_rgb_heldout_probability_histograms_512bins.npz'
assert HIST_PATH.exists(), 'Run updated Section 07 first to calculate probability histograms.'
with np.load(HIST_PATH,allow_pickle=False) as z:
    hist={k:z[k] for k in z.files}
def auc_ap_hist(keys):
    """V3_1-style binned ROC-AUC and approximate AP, 512 probability bins."""
    cs=[hist[f'fold{fold}_{name}_count'] for fold,name in keys]
    ps=[hist[f'fold{fold}_{name}_positive'] for fold,name in keys]
    if not cs:return np.nan,np.nan,0,0
    c=np.sum(cs,axis=0).astype(np.float64)
    p=np.sum(ps,axis=0).astype(np.float64)
    n=c-p;np_=p.sum();nn=n.sum()
    if np_==0 or nn==0:return np.nan,np.nan,int(np_),int(nn)
    # Rank AUC with half-credit for ties inside a bin.
    auc=float(np.dot(p,np.cumsum(n)-.5*n)/(np_*nn))
    cp=np.cumsum(p[::-1]);cn=np.cumsum(n[::-1])
    ap=float(np.sum((p[::-1]/np_)*(cp/np.maximum(cp+cn,1))))
    return auc,ap,int(np_),int(nn)
SITES=['Baego','MOS','Yangambi','Yoko']
def cover_stats(g,column):
    y=g.crown_reference_pct.to_numpy(float)
    p=g[column].to_numpy(float)
    err=p-y;n=len(y);rmse=float(np.sqrt(np.mean(err**2)))
    sd=float(np.std(y,ddof=1)) if n>1 else np.nan
    iqr=float(np.percentile(y,75)-np.percentile(y,25)) if n>1 else np.nan
    slope,intercept=(np.polyfit(y,p,1) if n>=2 and sd>1e-10 else (np.nan,np.nan))
    rr=float(np.corrcoef(y,p)[0,1]) if n>=2 and sd>1e-10 and np.std(p)>1e-10 else np.nan
    return dict(n_plots=n,cover_R2=float(r2_score(y,p)) if n>1 and sd>1e-10 else np.nan,
        cover_Pearson_r=rr,cover_MAE_pp=float(np.mean(np.abs(err))),
        cover_RMSE_pp=rmse,cover_bias_pp=float(np.mean(err)),
        RPD=float(sd/rmse) if rmse>0 else np.nan,
        RPIQ=float(iqr/rmse) if rmse>0 else np.nan,
        calibration_slope=float(slope),calibration_intercept_pp=float(intercept))
def group_stats(g,label,mode):
    col={'hard_mask':'predicted_cover_pct',
         'probability_mean':'predicted_probability_cover_pct'}[mode]
    z=cover_stats(g,col)
    tp,fp,fn,tn=(int(g[k].sum()) for k in ['tp','fp','fn','tn'])
    keys=[(int(v.fold),str(v.Plot_name)) for v in g.itertuples(index=False)]
    auc,ap,npos,nneg=auc_ap_hist(keys)
    z.update(group=label,cover_method=mode,
        n_rgb_reference_pixels=int(g.n_rgb_valid_label_pixels.sum()),
        n_positive_pixels=npos,n_negative_pixels=nneg,
        n_positive_plots=int((g.tp+g.fn>0).sum()),
        n_zero_positive_plots=int((g.tp+g.fn==0).sum()),
        pixel_ROC_AUC=auc,pixel_average_precision=ap,
        pixel_prevalence=float(npos/(npos+nneg)) if npos+nneg else np.nan,
        pixel_precision=float(tp/(tp+fp)) if tp+fp else np.nan,
        pixel_recall=float(tp/(tp+fn)) if tp+fn else np.nan,
        pixel_F1=float(2*tp/(2*tp+fp+fn)) if 2*tp+fp+fn else np.nan,
        pixel_IoU=float(tp/(tp+fp+fn)) if tp+fp+fn else np.nan,
        pixel_specificity=float(tn/(tn+fp)) if tn+fp else np.nan,
        pixel_balanced_accuracy=(.5*(tp/(tp+fn)+tn/(tn+fp))
            if tp+fn and tn+fp else np.nan),
        mean_Brier=float(np.average(g.brier_score,
              weights=g.n_rgb_valid_label_pixels)))
    return z
rows=[]
for mode in ['hard_mask','probability_mean']:
    rows.append(group_stats(R,'ALL',mode))
    for site in SITES:
        rows.append(group_stats(R[R.site==site],site,mode))
    for fold in sorted(R.fold.unique()):
        rows.append(group_stats(R[R.fold==fold],f'fold_{fold}',mode))
    for (fold,site),g in R.groupby(['fold','site']):
        rows.append(group_stats(g,f'fold_{fold}_{site}',mode))
METRICS=pd.DataFrame(rows)
METRICS.to_csv(REPORT/'02_v4_rgb_comprehensive_performance.csv',index=False)
display(METRICS.loc[METRICS.cover_method.eq('hard_mask') &
                    METRICS.group.isin(['ALL']+SITES)].round(3))
print('Soft probability fraction is a different prediction target from thresholded cover:')
display(METRICS.loc[METRICS.cover_method.eq('probability_mean') &
                    METRICS.group.isin(['ALL']+SITES)][
                    ['group','n_plots','cover_R2','cover_MAE_pp','cover_RMSE_pp',
                     'cover_bias_pp','RPD','RPIQ']].round(3))
print('Zero-positive plots: F1/recall undefined at plot level, but'
      ' included in false-positive, Brier and cover-error statistics.')
# Plot-wise scatter with 1:1 reference and site-specific points.
COLORS={'Baego':'#3168B4','MOS':'#E28E23','Yangambi':'#27856A','Yoko':'#AB477B'}
fig,axs=plt.subplots(1,2,figsize=(12.5,5.6),layout='constrained')
for ax,(method,col) in zip(axs,[
    ('Hard 0.5 mask','predicted_cover_pct'),
    ('Mean Musanga probability','predicted_probability_cover_pct')]):
    for site in SITES:
        d=R[R.site==site]
        ax.scatter(d.crown_reference_pct,d[col],s=52,alpha=.83,
                   label=f'{site} (n={len(d)})',color=COLORS[site],
                   edgecolors='white',linewidths=.5)
    ax.plot([0,100],[0,100],'--',color='.3',lw=1)
    cc=cover_stats(R,col)
    ax.set(xlim=(-2,103),ylim=(-2,103),aspect='equal',
           title=f'{method}: R²={cc["cover_R2"]:.3f}, MAE={cc["cover_MAE_pp"]:.2f} pp',
           xlabel='Observed Musanga fractional crown cover (%)',
           ylabel='Predicted Musanga cover (%)')
    ax.grid(alpha=.15)
axs[0].legend(loc='upper left',fontsize=8)
fig.savefig(REPORT/'02_plot_cover_observed_vs_predicted.png',dpi=230,bbox_inches='tight')
fig.savefig(REPORT/'02_plot_cover_observed_vs_predicted.pdf',bbox_inches='tight')
plt.show()
plt.close(fig)
fig,axs=plt.subplots(2,2,figsize=(9.5,9.5),sharex=True,sharey=True,layout='constrained')
for ax,site in zip(axs.ravel(),SITES):
    d=R[R.site==site]
    ax.scatter(d.crown_reference_pct,d.predicted_cover_pct,
         color=COLORS[site],s=55,alpha=.84,edgecolors='white',linewidths=.5)
    ax.plot([0,100],[0,100],'--',color='.3')
    mm=cover_stats(d,'predicted_cover_pct')
    ax.set(title=f'{site} (n={len(d)}): R²={mm["cover_R2"]:.3f}, MAE={mm["cover_MAE_pp"]:.2f} pp',
        xlim=(-2,103),ylim=(-2,103),aspect='equal')
    ax.grid(alpha=.14)
fig.supxlabel('Observed Musanga crown cover (%)')
fig.supylabel('Predicted Musanga crown cover (%)')
fig.savefig(REPORT/'02_per_site_plot_fraction_scatter.png',dpi=230,bbox_inches='tight')
fig.savefig(REPORT/'02_per_site_plot_fraction_scatter.pdf',bbox_inches='tight')
plt.show();plt.close(fig)
print('Full metric table and publication-quality plots:',REPORT)


## 06 — Full 34-circle held-out KVO validation (V3_1 method)
Replaces the misleading **clipped residual-circle** evaluation. Evaluate each of the original 34 KVO circles on its **full polygon footprint**, with its assigned held-out fold checkpoint and the KVO-corrected crown annotation reference. Preserve the V3_1 ROI and RGB-validity masks, but **do not subtract field-plot pixels**. Overlapping circles and plots were assigned to the same spatial fold in NB01, so this is a valid *secondary reference evaluation*, not 34 independent new geographic locations. Report footprint coverage, confusion statistics, binned ROC-AUC/AP, fractional-cover R²/RMSE/MAE/RPD/RPIQ and circle-labelled scatterplots. A separate subset identifies the historical 30-circle V3_1 cohort; model-to-model claims still require matching fold assignments and imagery supports.


### 09A — Mandatory consistency diagnostic before full-circle evaluation
The first full-circle evaluation gave implausibly fold-dependent predictions (fold 2 approximately 95–97% cover). **Run this short inference diagnostic first.** It compares two model-output calls on the same 256 px tiles, prints per-fold RGB values and predictions, and summarizes any existing circle output. No training and no full-circle inference.


In [ ]:
# 09A — Critical inference equivalence probe: BEFORE trusting circle R²
# Single tile (Section 09 style) MUST agree with Section 07 batch predictions.
# This uses 3 sites × 3 folds when available and runs on only 9 image tiles.
from rasterio.windows import Window
import numpy as np, torch
assert not SMOKE_TEST and set(RUN_FOLDS)=={0,1,2}
assert CIRCLE_ROI.circle_id.nunique()==34
rng=np.random.default_rng(20261113)
audit=[]
for fold in RUN_FOLDS:
    ck=torch.load(CHECKPOINTS[fold],map_location='cpu',weights_only=False)
    model=ResNet18UNet().to(DEVICE)
    model.load_state_dict(ck['model']);model.eval()
    rows=SPLITS[(SPLITS.cv_fold==fold)&(SPLITS.split=='val')]
    for site,g in rows.groupby('site'):
        q=g.sample(n=1,random_state=130+fold).iloc[0]
        mosaic=q.mosaic
        # Section 07 starts at the 512-window origin and subdivides it into 256 tiles.
        win=Window(int(q.col),int(q.row),min(CHIP,int(q.width)),min(CHIP,int(q.height)))
        with staged_mosaic(mosaic),torch.inference_mode():
            rgb,valid=rgb_window(RGB[mosaic]['path'],win)
            h,w=int(win.height),int(win.width)
            if h<CHIP or w<CHIP:
                rgb=np.pad(rgb,((0,0),(0,CHIP-h),(0,CHIP-w)),mode='constant')
            raw=torch.from_numpy(rgb[None]).to(DEVICE)
            with torch.autocast(device_type='cuda',dtype=torch.float16,enabled=AMP):
                p7=torch.sigmoid(model(raw))[:,0].float().cpu().numpy()[0,:h,:w]
                p9=torch.sigmoid(model(raw))[0,0].float().cpu().numpy()[:h,:w]
            discrepancy=float(np.max(np.abs(p7-p9)))
            assert discrepancy<=0.005,('Section 07 vs Section 09 DIFFER',fold,site,discrepancy)
            y,weight,_=supervision(mosaic,win,fold,'val')
            support=valid&weight.astype(bool)
            p=p9[support] if support.any() else p9[valid]
            audit.append(dict(fold=fold,site=site,mosaic=mosaic,
                row=int(win.row_off),col=int(win.col_off),
                n_heldout_plot_pixels=int(support.sum()),
                rgb_mean_0to1=float(rgb[:,:h,:w].mean()),
                rgb_max_0to1=float(rgb[:,:h,:w].max()),
                mean_probability=float(p.mean()) if p.size else np.nan,
                hard_cover_pct=float(100*(p>=.5).mean()) if p.size else np.nan,
                pixel_path_max_difference=discrepancy))
    del model;torch.cuda.empty_cache()
AUDIT=pd.DataFrame(audit)
display(AUDIT.round(4))
print('Same-pixel inference implementations agree. Note: these are sampled '
      'validation tiles and may have no plot reference (n_heldout_plot_pixels=0).')
# Also compare full-circle output against field-plot output on spatially overlapping
# SAME model fold; circle and plot cover naturally differ because support differs.
csv=OUT/'performance_report/circles_full_v3style/02_v4_full34_circles_by_circle.csv'
if csv.exists():
    d=pd.read_csv(csv)
    print('Already saved circle prediction fractions by fold:')
    display(d.groupby('fold').agg(n=('circle_id','size'),
       min_pred=('predicted_fraction','min'),
       median_pred=('predicted_fraction','median'),
       max_pred=('predicted_fraction','max'),
       mean_obs=('observed_fraction','mean')).round(2))
print('IMPORTANT: Do NOT interpret full-circle R² until diagnostics are resolved.')


In [ ]:
# 09 — V3_1-style COMPLETE circle evaluation: 34 held-out KVO circles, NO plot subtraction
assert not SMOKE_TEST and set(RUN_FOLDS)=={0,1,2}, (
    'Full three-fold checkpoints required. Execute Section 06B.')
from rasterio.windows import Window, from_bounds
from rasterio.features import rasterize
from shapely.geometry import box
from sklearn.metrics import r2_score
import matplotlib.pyplot as plt

FULL_CIRCLE_DIR=OUT/'performance_report'/'circles_full_v3style'
FULL_CIRCLE_DIR.mkdir(parents=True,exist_ok=True)
assert len(CIRCLE_ROI)==34 and CIRCLE_ROI.circle_id.nunique()==34
assert not CIRCLE_ROI.cross_fold_plot_intersection.any(), (
    'Circles with cross-fold proximity require explicit exclusion/diagnosis')
assert CIRCLE_ROI.cv_fold.isin(RUN_FOLDS).all()

# NB01 explicitly groups every overlapping plot-circle pair in the same fold.
for c in CIRCLE_ROI.itertuples(index=False):
    overlaps=P_T.query(c.geometry,predicate='intersects')
    contradictory=[str(PLOTS.iloc[int(j)].Plot_name) for j in overlaps
                   if PLOTS.iloc[int(j)].site_key==c.site_key and
                   int(PLOTS.iloc[int(j)].cv_fold)!=int(c.cv_fold) and
                   PLOTS.iloc[int(j)].geometry.intersection(c.geometry).area>0.01]
    assert not contradictory,(c.circle_id,'plot/circle leakage',contradictory)

def _raster(geoms,win,transform):
    geoms=[g for g in geoms if g is not None and not g.is_empty]
    if not geoms:
        return np.zeros((int(win.height),int(win.width)),dtype=bool)
    return rasterize(((g,1) for g in geoms),out_shape=(
        int(win.height),int(win.width)),transform=transform,
        fill=0,dtype='uint8').astype(bool)

def _binned_classification(pos_hist,total_hist):
    """Same ranking-based formula as V3_1 02B, with 512 rather than 256 bins."""
    p=pos_hist.astype(float)
    n=total_hist.astype(float)-p
    np_,nn=float(p.sum()),float(n.sum())
    if not np_ or not nn:return np.nan,np.nan
    auc=float(np.dot(p,np.cumsum(n)-0.5*n)/(np_*nn))
    cp=np.cumsum(p[::-1]);cn=np.cumsum(n[::-1])
    ap=float(np.sum((p[::-1]/np_)*cp/np.maximum(cp+cn,1)))
    return auc,ap

def _full_circle_row(circle,model,mosaic):
    """One physical circle, whole footprint; valid RGB and ROI masks follow V3_1."""
    cid=int(circle.circle_id)
    geom=circle.geometry
    ds=_ACTIVE_RASTER
    assert ds is not None and _ACTIVE_NAME==mosaic
    local=from_bounds(*geom.bounds,transform=ds.transform)
    rr0=max(0,int(np.floor(local.row_off/CHIP))*CHIP)
    cc0=max(0,int(np.floor(local.col_off/CHIP))*CHIP)
    rr1=min(ds.height,int(np.ceil((local.row_off+local.height)/CHIP))*CHIP)
    cc1=min(ds.width,int(np.ceil((local.col_off+local.width)/CHIP))*CHIP)
    sumy=sump_binary=sump_prob=brier_sum=0.
    tp=fp=fn=tn=n=0
    total_hist=np.zeros(N_BINS,dtype=np.int64)
    pos_hist=np.zeros(N_BINS,dtype=np.int64)
    geometry_support_pixels=rgb_valid_circle_pixels=0
    tile_count=0
    # A reference circle is counted exactly once on its selected mosaic.
    with torch.inference_mode():
        for rr in range(rr0,rr1,CHIP):
            for cc in range(cc0,cc1,CHIP):
                h=min(CHIP,ds.height-rr);w=min(CHIP,ds.width-cc)
                if h<=0 or w<=0:continue
                win=Window(cc,rr,w,h)
                transform=rasterio.windows.transform(win,ds.transform)
                footprint=box(*rasterio.windows.bounds(win,ds.transform))
                circle_mask=_raster([geom],win,transform)
                if not circle_mask.any():continue
                geometry_support_pixels+=int(circle_mask.sum())
                rgb,valid=rgb_window(RGB[mosaic]['path'],win)
                rgb_valid_circle_pixels+=int((circle_mask&valid).sum())
                roi_mask=_raster(geom_hits(ROI_T,ROI_GEOMS,footprint),win,transform)
                support=circle_mask&roi_mask&valid
                if not support.any():continue
                truth=_raster(geom_hits(KPOS_T,CIRCLE_POS_GEOMS,footprint),win,transform)
                if h<CHIP or w<CHIP:
                    rgb=np.pad(rgb,((0,0),(0,CHIP-h),(0,CHIP-w)),
                        mode='constant')
                with torch.autocast(device_type='cuda',dtype=torch.float16,enabled=AMP):
                    pred=torch.sigmoid(model(torch.from_numpy(
                        rgb[None]).to(DEVICE)))[0,0].float().cpu().numpy()[:h,:w]
                y=truth[support].astype(np.uint8)
                probs=pred[support].astype(np.float64)
                labels=probs>=0.5
                tp+=int(np.count_nonzero(labels&(y==1)))
                fp+=int(np.count_nonzero(labels&(y==0)))
                fn+=int(np.count_nonzero(~labels&(y==1)))
                tn+=int(np.count_nonzero(~labels&(y==0)))
                n+=len(y)
                sumy+=float(y.sum());sump_binary+=float(labels.sum())
                sump_prob+=float(probs.sum())
                brier_sum+=float(np.square(probs-y).sum())
                bins=np.minimum((probs*N_BINS).astype(np.int32),N_BINS-1)
                total_hist+=np.bincount(bins,minlength=N_BINS).astype(np.int64)
                pos_hist+=np.bincount(bins,weights=y,minlength=N_BINS).astype(np.int64)
                tile_count+=1
    assert int(total_hist.sum())==n and int(pos_hist.sum())==sumy
    pixel_area=abs(float(ds.transform.a*ds.transform.e-ds.transform.b*ds.transform.d))
    estimated_geom_pix=geom.intersection(box(*ds.bounds)).area/pixel_area
    status='complete_circle_covered' if n>0 else 'no_valid_roi_rgb'
    auc,ap=_binned_classification(pos_hist,total_hist)
    return dict(circle_id=cid,site=str(circle.site_key),
        fold=int(circle.cv_fold),mosaic=mosaic,status=status,
        complete_circle_area_m2=float(geom.area),
        circle_geometry_covered_by_selected_mosaic_pct=(
            100*geom.intersection(box(*ds.bounds)).area/geom.area),
        reference_support_pct_of_circle=100*n/max(estimated_geom_pix,1),
        circle_pixels_in_mosaic=geometry_support_pixels,
        rgb_valid_circle_pixels=rgb_valid_circle_pixels,
        valid_roi_rgb_reference_pixels=n,
        rgb_roi_reference_fraction_of_geometric_circle=(
            100*n/max(geom.area/pixel_area,1)),
        n_modelled_tiles=tile_count,
        observed_fraction=100*sumy/n if n else np.nan,
        predicted_fraction=100*sump_binary/n if n else np.nan,
        predicted_probability_fraction=100*sump_prob/n if n else np.nan,
        MAE_pp=100*abs(sump_binary-sumy)/n if n else np.nan,
        bias_pp=100*(sump_binary-sumy)/n if n else np.nan,
        Brier=brier_sum/n if n else np.nan,
        pixel_ROC_AUC=auc,pixel_AP=ap,
        pixel_F1=2*tp/(2*tp+fp+fn) if (tp+fn)>0 and (2*tp+fp+fn)>0 else np.nan,
        pixel_IoU=tp/(tp+fp+fn) if (tp+fn)>0 and (tp+fp+fn)>0 else np.nan,
        pixel_precision=tp/(tp+fp) if (tp+fp)>0 else np.nan,
        pixel_recall=tp/(tp+fn) if (tp+fn)>0 else np.nan,
        tp=tp,fp=fp,fn=fn,tn=tn)

# Select the highest-area RGB mosaic with positive circle intersection. Confirm
# this selection covers the full reference circle and audit any truncation.
CIRCLE_MOSAIC={}
for c in CIRCLE_ROI.itertuples(index=False):
    scores=[]
    for mosaic,info in RGB.items():
        if info['site']!=c.site_key:continue
        with rasterio.open(info['path']) as ds:
            coverage=float(c.geometry.intersection(box(*ds.bounds)).area)
        if coverage>0:scores.append((coverage,mosaic))
    assert scores,(c.circle_id,'No RGB mosaic intersects held-out circle')
    CIRCLE_MOSAIC[int(c.circle_id)]=max(scores)[1]

CIRCLE_RECORDS=[]
for fold in RUN_FOLDS:
    ck=torch.load(CHECKPOINTS[fold],map_location='cpu',weights_only=False)
    assert ck['fold']==fold and ck.get('smoke') is False
    model=ResNet18UNet().to(DEVICE)
    model.load_state_dict(ck['model'])
    model.eval()
    fc=CIRCLE_ROI.loc[CIRCLE_ROI.cv_fold.eq(fold)]
    for mosaic in sorted(set(CIRCLE_MOSAIC[int(i)] for i in fc.circle_id)):
        group=fc.loc[fc.circle_id.map(CIRCLE_MOSAIC).eq(mosaic)]
        with staged_mosaic(mosaic):
            for circle in group.itertuples(index=False):
                result=_full_circle_row(circle,model,mosaic)
                CIRCLE_RECORDS.append(result)
                print('FULL CIRCLE',result['circle_id'],result['site'],
                    'fold',fold,'valid pixels',result['valid_roi_rgb_reference_pixels'],
                    'RGB footprint %',round(result['circle_geometry_covered_by_selected_mosaic_pct'],1),
                    'obs/pred',round(result['observed_fraction'],1),
                    round(result['predicted_fraction'],1),flush=True)
    del model
    if AMP:torch.cuda.empty_cache()

CIRCLE_SCORES=pd.DataFrame(CIRCLE_RECORDS).sort_values('circle_id')
assert len(CIRCLE_SCORES)==34 and CIRCLE_SCORES.circle_id.is_unique
assert CIRCLE_SCORES.valid_roi_rgb_reference_pixels.gt(0).all(),(
    'At least one circle has no valid pixels; inspect CSV and mosaic coverage')
CIRCLE_SCORES.to_csv(FULL_CIRCLE_DIR/'02_v4_full34_circles_by_circle.csv',index=False)
display(CIRCLE_SCORES[['circle_id','site','fold','mosaic','observed_fraction',
    'predicted_fraction','rgb_roi_reference_fraction_of_geometric_circle']].round(2))

# Full-vs-partially-covered circles must be interpreted separately; never make
# an unlabelled assumption that all 34 footprints are complete.
coverage_limit=95.
covered=CIRCLE_SCORES.loc[
    CIRCLE_SCORES.rgb_roi_reference_fraction_of_geometric_circle>=coverage_limit]
if len(covered)<34:
    print(f'ATTENTION: {34-len(covered)} circle(s) have <{coverage_limit}% '
          'valid RGB/ROI support; see per-circle audit and restricted metrics.')

def summarise_circle(g,site,cohort):
    y=g.observed_fraction.to_numpy(float)
    p=g.predicted_fraction.to_numpy(float)
    error=p-y
    n=len(g)
    rmse=float(np.sqrt(np.mean(error**2)))
    sd=float(np.std(y,ddof=1)) if n>1 else np.nan
    iqr=float(np.percentile(y,75)-np.percentile(y,25)) if n>1 else np.nan
    return dict(cohort=cohort,site=site,n_circles=n,
        R2=float(r2_score(y,p)) if n>1 and sd>1e-10 else np.nan,
        Pearson_r=float(np.corrcoef(y,p)[0,1]) if n>1 and sd>1e-10 and np.std(p)>0 else np.nan,
        MAE_pp=float(np.mean(abs(error))),RMSE_pp=rmse,
        bias_pp=float(np.mean(error)),RPD=sd/rmse if rmse>0 else np.nan,
        RPIQ=iqr/rmse if rmse>0 else np.nan,
        mean_pixel_F1=float(g.pixel_F1.mean()),
        mean_pixel_IoU=float(g.pixel_IoU.mean()),
        mean_pixel_ROC_AUC=float(g.pixel_ROC_AUC.mean()),
        mean_pixel_AP=float(g.pixel_AP.mean()))
def cohort_summary(g,cohort):
    if g.empty:return []
    return [summarise_circle(g,'ALL',cohort)]+[
        summarise_circle(z,site,cohort) for site,z in g.groupby('site')]
metrics=cohort_summary(CIRCLE_SCORES,'full34_RGB_ROI')
metrics+=cohort_summary(covered,'full_circle_coverage_ge95pct')

# Identify the old 30-circle cohort exactly from the V3_1 reference.
V31_LINK=ROOT/'Outputs/V3_1/01_reference_data_no_MOS34/01_circle_spatial_folds.csv'
if V31_LINK.exists():
    historic=pd.read_csv(V31_LINK)
    old_ids=set(pd.to_numeric(historic.circle_id).astype(int))
    common=CIRCLE_SCORES[CIRCLE_SCORES.circle_id.isin(old_ids)]
    assert len(old_ids)==30 and len(common)==30 and common.circle_id.nunique()==30
    metrics+=cohort_summary(common,'V3_1_original_30_ids_V4_folds')
    common.to_csv(FULL_CIRCLE_DIR/'02_v4_historical_30_circle_ids.csv',index=False)
    print('Historical V3_1 circle ID set identified. Different V4 folds/reference: '
          'descriptive only until exact pairing is established.')
else:
    print('V3_1 circle-fold CSV absent: not guessing the historical 30 IDs.')

CIRCLE_SUMMARY=pd.DataFrame(metrics)
CIRCLE_SUMMARY.to_csv(FULL_CIRCLE_DIR/'02_v4_full_circle_summary.csv',index=False)
display(CIRCLE_SUMMARY.round(3))
fig,ax=plt.subplots(figsize=(8,7),layout='constrained')
colors={'Baego':'#3168B4','MOS':'#E28E23','Yangambi':'#27856A','Yoko':'#AB477B'}
for site,g in CIRCLE_SCORES.groupby('site'):
    ax.scatter(g.observed_fraction,g.predicted_fraction,color=colors[site],
               label=f'{site} (n={len(g)})',s=60,alpha=.85)
    for r in g.itertuples(index=False):
        ax.annotate(str(r.circle_id),(r.observed_fraction,r.predicted_fraction),
                    xytext=(3,3),textcoords='offset points',fontsize=7)
ax.plot([0,100],[0,100],'--',color='.35')
ax.set(xlim=(-2,103),ylim=(-2,103),aspect='equal',
    xlabel='Observed Musanga cover (%) — KVO circle annotations',
    ylabel='V4 predicted Musanga cover (%) — threshold 0.5',
    title='Full-circle V4 RGB evaluation, three spatially held-out folds')
ax.grid(alpha=.15);ax.legend(frameon=False)
fig.savefig(FULL_CIRCLE_DIR/'02_v4_full34_observed_predicted.png',
            dpi=220,bbox_inches='tight')
fig.savefig(FULL_CIRCLE_DIR/'02_v4_full34_observed_predicted.pdf',
            bbox_inches='tight')
plt.show();plt.close(fig)
print('Full-circle V3_1-style benchmark saved:',FULL_CIRCLE_DIR)
